# Objetivos

* **MÓDULOS PERDIDOS**: Reportar sobre los datasets ligados a un diagnóstico específico cuya composición es 100% nulo, es decir, que no se realizaron o perdieron todos sus datos por algún error humano o de sistema.
* **DATASETS DE COMPOSICIÓN**: Documentar el significado de las columnas dentro de los archivos de composiciones de preguntas dentro de la carpeta *Python/resultados/composiciones*.
* **REPORTE DE PREGUNTAS BASALES**: Reportar las preguntas basales encontradas y las razones para su determinación.

In [2]:
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
from pathlib import Path

import utils as ut

In [3]:
df_sintomas_disc = pd.read_excel(r"C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\data\Sintomas_disc_Infante_Juvenil_n=1558.xlsx")

In [4]:
full_dict = ut.build_full_dict(df_sintomas_disc)

# MÓDULOS PERDIDOS

Ahora se presentan los módulos cuya composición es 100% nula.

In [6]:
rows = []

for code, info in full_dict.items():
    full_nan = (info['df_comp_full']['pct_nan'] == 100).all()
    rows.append({
        "code": code,
        "Nombre": info["nombre"],
        "Módulo": info["modulo"],
        "Perdido": full_nan
    })

df_result = pd.DataFrame(rows)
display(df_result)

,code,Nombre,Módulo,Perdido
0,pag,Agorafobia,A,True
1,psp,Fobia específica,A,True
2,pso,Fobia social,A,False
3,pga,Ansiedad generalizada,A,False
4,psm,Mutismo selectivo,A,True
5,ppa,Pánico,A,True
6,ppt,Trastorno por estrés postraumático,A,True
7,psa,Ansiedad por separación,A,False
8,poc,Trastorno obsesivo-compulsivo,A,True
9,pea,Bulimia,B,False


# DATASETS DE COMPOSICIÓN

Los siguientes datasets de composición, guardados en .csv y .xlsx dentro de la carpeta Python/resultados/composiciones cuentan con las siguientes columnas:

| Nombre de Columna | Descripción |
|-----------|-----------|
|      question     |  Código de la pregunta con la forma estándar *(cod)(dig)(abc)*, donde *(cod)* es el código correspondiente a cada diagnóstico, *(dig)* son 3 dígitos ordinales y *(abc)* son caracteres opcionales que indican preguntas de seguimiento.        |
|     cnt_non_nan      |    Conteo total de respuestas no perdidas de un total de N=1558       |
|     pct_non_nan      |    Prcentaje total de respuestas no perdidas de un total de N=1558       |
|cnt_non_standard|Conteo total de respuestas distintas al conjunto {2, 0, 3, 7, 77, 8, 88, 9, 99 o NaN} de un total de N=1558|
|pct_non_standard|Porcentaje total de respuestas distintas al conjunto conjunto {2, 0, 3, 7, 77, 8, 88, 9, 99 o NaN} de un total de N=1558|
|cnt_*| Conteo único para la respuesta *, incluyendo al conjunto {2, 0, 3, 7, 77, 8, 88, 9, 99 o NaN}. Para el subconjunto {2, 0, 3, 7, 77, 8, 88, 9 y 99} se agrega una aclaración sobre el significado de la respuesta estándar, ver abajo. |
|cnt_*| Porcentaje para la respuesta * de un total de N=1558, incluyendo al conjunto {2, 0, 3, 7, 77, 8, 88, 9, 99 o NaN}. Para el subconjunto {2, 0, 3, 7, 77, 8, 88, 9 y 99} se agrega una aclaración sobre el significado de la respuesta estándar, ver abajo. |

Respuestas estándar:

|Numéricamente| Significado| Código|
|---|---|---|
|2|Sí|_SI|
|0|No|_NO|
|3| A veces / De alguna manera| _AV/A|
|7, 77| Se niega a respoder| _NR|
|8, 88| No aplica| _NA|
|9, 99| No sabe| _NS|

In [8]:
save_done=False

if save_done == True:
    out_dir = Path.cwd() / "resultados" / "composiciones" / "csv"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for code, info in full_dict.items():
        info["df_comp_full"].to_csv(out_dir / f"{code}_comp.csv", index=False)

In [9]:
if save_done == True:
    out_dir = Path.cwd() / "resultados" / "composiciones" / "xlsx"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for code, info in full_dict.items():
        info["df_comp_full"].to_excel(out_dir / f"{code}_comp.xlsx", index=False)

# REPORTE DE PREGUNTAS BASALES

La metodología para encontrar las preguntas basales sin revisar el contenido de ninguna de ellas es sencillo: De forma efectiva, sólo se ordenan en razón de sus nulos, se asume una "tolerancia" y se encuentran umbrales de corte, donde la nulidad se dispara. En este sentido la tolerancia refleja el porcentaje máximo (excluyente) asumido de preguntas basales para un conjunto de preguntas desconocidas; y los umbrales son el máximo porcentaje de valores nulos dentro de una pregunta para ser considerada basal. A menor tolerancia, se obtienen menores umbrales.

A continuación, para cada módulo, se reportan las preguntas con menores nulos, según esta metodología. Además, al hacer la revisión dentro de cada módulo, se comprobaron los "cortes", es decir aquellos quiebres en la continuidad numérica de las pregutas. Si se reportan como basales las preguntas 1, 2, 4 y 5, dentro de un conjunto donde se tienen preguntas de la 1 a la 10, sería necesario evaluar si luego de la pregunta 2 se da alguna indicación dentro del cuestionario que permita saltar la pregunta 3 y, luego de la 5, saltar al siguiente módulo. De esta forma se encontraría el mínimo número de preguntas que deben ser respondidas.

Éste análisis es claramente perfectible y podría requerir de análisis experto.

En el anexo es posible encontrar las preguntas para las primeras 3 menores tolerancias.

## Fobia Social - pso

**Basales: 1-5**

Luego de la pregunta 5 se da la siguiente indicación: 


>a:	SI SE CODIFICO ALGUNA RESPUESTA CON * EN LAS P 3 - 5, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A TASEP, PAG.  9

Es decir, que el output es correcto sólo pso001-5 son basales

---

## Ansiedad generalizada - pga

**Basales: 1-5, 23-29**

En este caso hay una anomalía, ya que hay más datos perdidos en pga001 que en pga002-5 y pga023-29. aún así, estos resultados tienen sentido, luego de 5 se da la instrucción:

>h:	SI SE CODIFICARON 1 O MAS RESPUESTAS CON * EN LAS P 1-5, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A LA P23 , PAG. 56

Y  luego de la 29:

>m:	SI SE CODIFICO UNA RESPUESTA CON π EN LA P6, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A MUSE, PAG. 59

---

## Ansiedad por separación - psa

**Basales: 1-12**

Luego de la pregunta 1 se indica:

>	a:	SI EL NIÑO NO ASISTIO A LA ESCUELA NI AL TRABAJO EN EL ULTIMO AÑO, CODIFIQUE "8" EN LA P 2 Y PASE A LA P 3.

Lo que indica que algunos datos que debían codificarse como *8* se codificaron como NaN, o que, genuinamente, se perdieron por el sistema. Además, luego de la pregunta 12, se indica:

> d:	SI SE CODIFICARON 2 O MAS RESPUESTAS CON [ ] EN LAS P 1 - 12 Y EN LAS NOTAS 1 -  4, CONTINUE
>
>	CUALQUIER OTRA RESPUESTA, PASE A FOESP EN LA PAG. 19

---

## Bulimia - pea

**Basales: 1-4, 10-12, 20**

Es importante notar de los códigos que, dado que se pide medidas en distintos sistemas, la letra tras el número refleja la medida utilizada, por ejemplo, en la pregunta 2 se tiene *pea002k* y *pea002l*, que representan pesos en kilos y libras, respectivamente, pero ambas apuntan a la pregunta 2.

Parece haber una mayor cantidad de datos no codificados en la pregunta 1 que en el resto de preguntas basales.

Luego, en cuanto a la continuidad, tras la pregunta 4 se indica:

>NOTA 4:	¿SE CODIFICO ALGUNA RESPUESTA CON * EN LA NOTA 3 O LA P 4? 
>
>SI DICE SI:	CONTINUE
>SI DICE NO:	PASE A LA P10

Luego de la pregunta 12

>b.	SI SE CODIFICO ALGUNA RESPUESTA CON { } EN LA NOTA 4 O UNA RESPUESTA CON π EN P 12D, CONTINUE
>
>CUALQUIER OTRA RESPUESTA, PASE AL RECUADRO DE INSTRUCCIONES “d”, PAG. 9

El cuadro d indica:

>d: 	SI ES VARON, CODIFIQUE “8" EN LA P 20 Y PASE AL RECUADRO DE INSTRUCCIONES “e”, PAG. 11
>
>		CUALQUIER OTRA RESPUESTA, CONTINUE

Luego de la pregunta 20:

>  e:	SI SE CODIFICO { } EN LA NOTA 4 Y SE CODIFICARON UNA O MAS RESPUESTAS CON <> EN LA P 10-18 (Vea hoja de registro), PASE A P 21
>
>	CUALQUIER OTRA RESPUESTA, PASE AL RECUADRO DE INSTRUCCIONES “f”

Luego:

>f:	SI SE CODIFICARON 2 O MAS RESPUESTAS CON [ ] EN LAS  P10-19 Y LA NOTA 5, PASE A P22.
>
>		CUALQUIER OTRA RESPUESTA, PASE A ELIM, PAG 17

---


## Depresión mayor o distimia - pmd

**Basales: 1-22, 35**

Luego de la pregunta 22 se indica:

>NOTA 10:	¿SE CODIFICARON TRES O MAS RESPUESTAS CON [ ] EN LA P2 Y EN LAS NOTAS 2-9?
>
>SI DICE SI:	CONTINUE
>SI DICE NO:	PASE A LA P35, PAG. 21

Luego, en la pregunta 35 se da un condicional dentro de sus preguntas contingentes:

>   f:	SI SE CODIFICO ALGUNA RESPUESTA CON *  EN LA P 35B, PASE A LA P 37
>
>	CUALQUIER OTRA RESPUESTA, CONTINUE

Pero la pregunta 37 no es basal porque en la pregunta 36 se indica:

>   g:	SI SE CODIFICO ALGUNA RESPUESTA CON * EN LA P36B, CONTINUE
>
>CUALQUIER OTRA RESPUESTA, PASE A MAN/HIPOMAN, PAG. 29

---

## Manía o hipomanía - pma

**Basales: 1-13**

En la pregunta, 13, previo a contestar la pregunta 14 se indica:

>  o:	SI NO SE CODIFICO ALGUNA RESPUESTA CON * EN LAS P 1-2, PASE A LA NOTA 1
>
>CUALQUIER OTRA RESPUESTA, CONTINUE

Mientras que la nota 1 indica:

>  NOTA 1:	¿SE CODIFICARON 2 O MAS RESPUESTAS CON [ ] EN LAS P 3 - 14?
>
>SI DICE SI:		CONTINUE
>SI DICE NO:	PASE AL MODULO D

---

## Esquizofrenia - psz

**Basales: 1-2, 9-14**

Luego de la pregunta 2, si se contesta "No", se indica:

>  a:	SI SE CODIFICO ALGUNA RESPUESTA CON [ ] EN LAS NOTAS 1 - 3 Y P 3, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A LA P 9, PAG. 6

Y luego de la 14:

> NOTA 17:		ESTE ES UN PATRON DE SALTO OPCIONAL
>
>ENTREVISTADOR:	¿SE CODIFICO ALGUNA RESPUESTA CON π EN LAS P 1 - 14 O EN LA NOTA 4?
>
>SI DICE SI:		CONTINUE
>SI DICE NO:	PASE AL PROXIMO   MODULO
>(SI PROCEDE EN EL SONDEO DE ESQUIZOFRENIA)

---

## Trastorno por déficit de atención e hiperactividad - pad

**Basales: 1-2, 4-11, 22-25, 27-29, 31-33, 44**

Luego de la pregunta 2d, si se contesta "Sí", se pasa a la Nota 1, posterior a la pregunta 3, luego, al llegar a la pregunta 11 se indica:

>	a.	SI SE CODIFICARON 3 O MAS RESPUESTAS CON [  ] EN LAS    P 1-10 Y LA NOTA 1 (hoja de registro), CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A LA P 22, PAG. 10	

Luego de la pregunta 25d, si se contesta "Sí", se pasa a la Nota 2, posterior a la pregunta 26 y lo mismo pasa para la preguntas 29d, Nota 3 y pregunta 30, respectivamente. En la pregunta 33d al ser respondida con con "No", se pasa al cuadro de instrucciones g:

>g:	SI SE CODIFICARON 3 O MAS RESPUESTAS CON [ ] EN LAS P 22 - 32 Y EN LAS NOTAS DE LA 2 - 3 (vea hoja de registro), CONTINUE
>
>CUALQUIER OTRA RESPUESTA, PASE A LA P 44, PAG. 18

Luego de la pregunta 44, se indica:
								
>	m:	SI SE CODIFICARON 4 O MAS RESPUESTAS CON [ ] EN LAS P 1-32 Y EN LAS NOTAS DE LA 1 - 3 O ALGUNA RESPUESTA CON † EN LA P 44 ) (vea hoja de registro), CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A TOD, PAG. 21

---
                                
## Trastorno de conducta - pcd

**Basales: 1-4, 7-9, 12-15, 19-23, 25, 27-29, 39**

Las preguntas 5 y 6 sólo se responden si hay respuestas con * en las preguntas 1-4. Lo mismo ocurre con las preguntas 10-11 dependiendo de las respuestas de 7-9 y con las preguntas 16-17, dependiendo de las respuestas de 14-15. En el caso de la pregunta 18, ocurre lo siguiente:

>a:	SI EL NIÑO NUNCA ASISTIO A LA ESCUELA NI TRABAJO, CODIFIQUE “8" EN LA P 18, LUEGO PASE A LA P 19

Con lo quue es probable que los valores que debieron ser *8* se codificaron como NaN. En la pregunta 24 se indica:

> 
PREGUNTA OPCIONAL	
>
>SOLO HAGA LA PREGUNTA DE ESTE RECUADRO SI HA SIDO APROBADA PARA SU ESTUDIO Y PARA LA EDAD DEL NIÑO.
>
>SI NO SE HACE ESTA PREGUNTA, CODIFIQUE "8" EN LA P24, Y LUEGO PASE A LA P25	
	
Con lo que la preunta 24 no se considera Basal. La pregunta 26 es contigente pues luego de la 25 se indica:

>  d:	SI SE CODIFICO ALGUNA RESPUESTA CON * Y CON † EN LA P 25, PASE A LA P 27
>
>	CUALQUIER OTRA RESPUESTA, CONTINUE

Luego de la pregunta 29, se indica:

>  f:	SI SE CODIFICARON 2 ó MAS RESPUESTAS CON [ ] EN LAS P 1-29 Y LAS NOTAS DE LA 1-5 (vea hoja de registro) CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE AL RECUADRO DE INSTRUCCIONES “h”, PAG. 61

El recuadro de instrucciones h indica:

>h:	SI EL NIÑO NUNCA ASISTIO A LA ESCUELA, CODIFIQUE “8" EN LAS P 36, P 37 Y P 38, LUEGO PASE A LA P 39

Luego de la 39, se indica:

>SI DICE NO, PASE AL RECUADRO DE INSTRUCCIONES “p”, PAG.58
		
Éste dice:	
	
>	p:	SI EL NIÑO NUNCA TRABAJO, CODIFIQUE “8" EN LA P 40, LUEGO PASE AL RECUADRO DE INSTRUCCIONES “q”
	
Luego:

>q:	SI SE CODIFICARON 2 O MAS RESPUESTAS CON [ ] EN LAS P 1 - 29 Y EN LAS NOTAS 1 - 5 (ver hoja de registro), CONTINUE
>
>	CUALQUIER OTRA RESPUESTA, PASE AL RECUADRO DE INSTRUCCIONES  “r”	

Luego:

>r:	SI SE CODIFICARON 3 O MAS RESPUESTAS CON [ ] EN LAS  
>P1-29 Y EN LAS NOTAS 1-5 (vea hoja de registro), PASE AL   MODULO F
>
>SI SE CODIFICARON 3 O MAS RESPUESTAS CON ‹ ›, CONTINUE
>     
>CUALQUIER OTRA RESPUESTA, PASE AL MODULO F

---

## Trastorno de oposición desafiante - pod

**Basales: 1-12**

Luego de la pregunta 12 se indica:

>g:		SI SE CODIFICARON 2 O MAS RESPUESTAS CON [ ] EN LAS P1-11 Y EN LAS NOTAS 1-3, CONTINUE
>
>					CUALQUIER OTRA RESPUESTA, PASE A TCD, PAG. 41

---

## Abuso de alcohol - pal

**Basal: 1**

Naturalmente, la única pregunta basal es la primera, ya que si no se ha probado, no es necesario seguir preguntando.

>SI DICE NO, PASE A TAB, PAG. 13

## Consumo de marihuana - pmj

**Basal: 1**

Mismo caso que con el alcohol.

>SI DICE NO, PASE A OTRAS SUBST., PAG. 37

---

## Consumo de otras sustancias - psu

**Basales: 1-11**

Luego de la pregunta 11, se indica:

>l:	SI HAY ALGUNA RESPUESTA CON “π” EN LAS P 1-11, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A LA NOTA 5, PAG. 61

Luego:

> 
>NOTA 5: 	SE CODIFICO ALGUNA RESPUESTA CON π O † EN P1-11?
>
>SI DICE SI:	CONTINUE
>SI DICE NO:	PASE AL MODULO TODA LA VIDA	

---

## Consumo de tabaco - pni

**Basales: 1-2**

Luego de la pregunta 2:

>SI DICE NO, PASE AL RECUADRO DE INSTRUCCIONES "c"

Luego:

>c.	SI SE CODIFICO  CON * LA RESPUESTA EN LA P  1B O LA P 2B, CONTINUE
>
>		CUALQUIER OTRA RESPUESTA, PASE A LA NOTA 3, PAG. 21 

Finalmente:

>NOTA 3:	¿SE CODIFICO ALGUNA  RESPUESTA CON  † EN LA P1 ó LA 2?
>
>SI DICE SI:	CONTINUE
>SI DICE NO:	PASE A MARIH PAG. 25

---

# Anexo

In [11]:
ut.tol_report(full_dict, 3)

Agorafobia - pag - N=64


------------------------------------------------------------------------------------------------------------------------------------------------------


Fobia específica - psp - N=70


------------------------------------------------------------------------------------------------------------------------------------------------------


Fobia social - pso - N=55
Tolerancia de 10% => Umbral de 0.06 a 43.97


,question,pct_nan,pct_0.0,pct_2.0,pct_3.0,pct_7.0,pct_8.0,pct_9.0,pct_88.0,pct_99.0,limited
0,pso001,0.064185,4.043646,95.892169,0.0,0.0,0.0,0.000000,0.0,0.0,True
2,pso002,0.064185,24.326059,75.609756,0.0,0.0,0.0,0.000000,0.0,0.0,True
3,pso003,0.064185,67.907574,32.028241,0.0,0.0,0.0,0.000000,0.0,0.0,True
5,pso004,0.064185,78.818999,20.924262,0.0,0.0,0.0,0.192555,0.0,0.0,True
6,pso005,0.064185,51.155327,48.652118,0.0,0.0,0.0,0.128370,0.0,0.0,True


Tolerancia de 13% => Umbral de 0.06 a 54.17


,question,pct_nan,pct_0.0,pct_2.0,pct_3.0,pct_7.0,pct_8.0,pct_9.0,pct_88.0,pct_99.0,limited
7,pso006,43.966624,12.451861,43.388960,0.0,0.0,0.0,0.192555,0.0,0.0,True
8,pso007,43.966624,37.355584,18.549422,0.0,0.0,0.0,0.128370,0.0,0.0,True


Tolerancia de 24% => Umbral de 54.17 a 67.01


...

Tolerancia de 6% => Umbral de 0.13 a 85.49


,question,pct_nan,pct_0.0,pct_2.0,pct_3.0,pct_7.0,pct_8.0,pct_9.0,pct_99.0,limited
67,pni028,83.889602,11.232349,4.878049,0.0,0.0,0.0,0.0,0.0,True


Tolerancia de 9% => Umbral de 82.09 a 90.50


,question,pct_nan,pct_0.0,pct_2.0,pct_3.0,pct_7.0,pct_8.0,pct_9.0,pct_99.0,limited
4,pni001c,85.494223,4.813864,9.691913,0.0,0.0,0.0,0.0,0.0,True
12,pni003,85.494223,5.006418,9.499358,0.0,0.0,0.0,0.0,0.0,True




------------------------------------------------------------------------------------------------------------------------------------------------------


